# Chapter 6: the cellar quarter, vessel by vessel

Builds the Kestrel quarter table in chapter 6.2 and Figure 6.1 from the minimum dataset. Kestrel uses unitanks, so the steps sit inside one vessel:

- fermentation: fermentation `volume_in_hl` less `volume_out_hl` (crop and cone dumps)
- cold conditioning: conditioning in less out, less the heel
- heel: `heel_hl` on the conditioning rows
- transfer to filter: conditioning out less filter in

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/minimum_dataset.csv")
w = df[df.stage != "cast"].pivot(index="batch_id", columns="stage", values=["volume_in_hl", "volume_out_hl", "heel_hl"])
vessel = df[df.stage == "fermentation"].set_index("batch_id")["vessel_id"]

steps = pd.DataFrame({
    "Fermentation: crop, cone dumps": w.volume_in_hl.fermentation - w.volume_out_hl.fermentation,
    "Cold conditioning: cone dumps, dry hop slurry": w.volume_in_hl.conditioning - w.volume_out_hl.conditioning - w.heel_hl.conditioning,
    "Heel left when the tank is emptied": w.heel_hl.conditioning,
    "Transfer to filter": w.volume_out_hl.conditioning - w.volume_in_hl["filter"],
})
cast = w.volume_in_hl.fermentation.sum()
print(f"Quarter cast {cast:,.0f} hL in {len(w)} fills")

## The quarter table (chapter 6.2)

In [ ]:
NORMAL_PCT = [1.00, 0.32, 0.30, 0.30]  # set from a quarter of well run batches; Kestrel's are synthetic
loss = steps.sum().round(1)
vol_in = cast - loss.cumsum().shift(fill_value=0)
q = pd.DataFrame({"Volume in (hL)": vol_in.round(0), "Loss (hL)": loss.round(0), "Loss, percent": 100 * loss / cast,
                  "Normal, percent": NORMAL_PCT})
q["Normal (hL)"] = cast * q["Normal, percent"] / 100
q["Excess (hL)"] = (q["Loss (hL)"] - q["Normal (hL)"]).round(0).clip(lower=0)
total = pd.DataFrame([[cast, loss.sum(), 100 * loss.sum() / cast, sum(NORMAL_PCT), cast * sum(NORMAL_PCT) / 100,
                       round(loss.sum() - cast * sum(NORMAL_PCT) / 100)]], columns=q.columns, index=["Total"])
q = pd.concat([q, total])
q.round(2)

In [ ]:
excess_hl = q.loc["Total", "Excess (hL)"]
to_filter = cast - loss.sum()
print(f"Reached the filter: {to_filter:,.0f} hL. Excess {excess_hl:.0f} hL = {100 * excess_hl / cast:.2f} percent of cast")
print(f"Wort at 24.00 a hL carries {24 / (to_filter / cast):.2f} per good hL at the filter")
e = q["Excess (hL)"]
step_value = e.iloc[0] * 27.00 + (e.iloc[1] + e.iloc[2]) * 29.50
print(f"Step by step value: USD {step_value:,.2f} a quarter, USD {4 * step_value:,.0f} a year")
year_hl = 4 * excess_hl
print(f"A year: {year_hl:.0f} hL excess. Replacement cost USD {year_hl * 30:,.0f}; upper bound USD {year_hl * 110:,.0f}")

## The same excess by vessel

If every fermenter runs high, the cause is a procedure. If one runs high, look at that vessel.

In [ ]:
by_vessel = steps.sum(axis=1).groupby(vessel).sum().to_frame("Loss (hL)")
by_vessel["Cast (hL)"] = w.volume_in_hl.fermentation.groupby(vessel).sum()
by_vessel["Loss, percent"] = 100 * by_vessel["Loss (hL)"] / by_vessel["Cast (hL)"]
by_vessel["Excess over 1.92 percent (hL)"] = by_vessel["Loss (hL)"] - by_vessel["Cast (hL)"] * 0.0192
by_vessel.sort_values("Excess over 1.92 percent (hL)", ascending=False).round(1)

In [ ]:
rows = q.iloc[:4]
labels = ["Fermentation", "Cold conditioning", "Heel left in tank", "Transfer to filter"]
fig, ax = plt.subplots(figsize=(7, 3))
normal = rows["Normal (hL)"].round(0)
ax.barh(labels, normal, color="#cfe6e1", edgecolor="#0b4f45", label="Normal loss for the step")
ax.barh(labels, rows["Excess (hL)"], left=normal, color="#b23a2e", label="Excess over normal")
for i, (n, x) in enumerate(zip(normal, rows["Excess (hL)"])):
    ax.text(n + x + 4, i, f"{x:.0f} hL excess", va="center")
ax.invert_yaxis()
ax.set_xlabel("Cellar loss in the quarter (hL)")
ax.set_title("Figure 6.1: loss by step, Kestrel quarter (synthetic)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()